In [ ]:
### PACKAGES

import qutip as qt
import numpy as np
import cvxpy as cp

In [ ]:
### QUANTUM CHANNELS

#Evolution of quantum state via Choi matrix
def evol(J, rho, correction = True):

    out = (qt.tensor(qt.qeye(J.dims[0][0]), rho.trans()) * J).ptrace(1)
    
    if correction is True:
        # 1. Force Hermiticity (makes all eigenvalues strictly real)
        out = 0.5 * (out + out.dag())
        
        # 2. Force positive semi-definiteness (makes all eigenvalues strictly non-negative)
        evals, evecs = np.linalg.eigh(out.full())
        evals = np.maximum(evals, 0)
        out = qt.Qobj(evecs @ np.diag(evals) @ evecs.conj().T, dims=out.dims)

        # 3. Force unit-trace (makes trace strictly equal one)
        out = out / out.tr()
    
    return out

# Fully depolarising channel
def J_depolarising(dy, dx):
    
    return qt.tensor(qt.qeye(dy), qt.qeye(dx))/dy

# Replacement channel for pure state
def J_replacement(dy, dx):
    
    rho_0 = qt.basis(dy, 0) * qt.basis(dy, 0).dag()
    return qt.tensor(rho_0, qt.qeye(dx))

# Fully dephasing channel
def J_dephasing(dy, dx):
        
    J = qt.Qobj(np.zeros((dy * dx, dy * dx)), dims=[[dy, dx], [dy, dx]])
    for i in range(dy):
        out_state = qt.basis(dy, i) * qt.basis(dy, i).dag()
        in_state = qt.basis(dx, i) * qt.basis(dx, i).dag()
        J += qt.tensor(out_state, in_state)
    return J

# Identity channel
def J_identity(dy, dx):
        
    # Vectorisation of the identity operator sum_i |i> \otimes |i>
    vec_I = sum([qt.tensor(qt.basis(dy, i), qt.basis(dx, i)) for i in range(dy)])
    return vec_I * vec_I.dag()

# Compression channel
def J_compression(dx, r, Pi=None, rho0=None):
    if Pi is None:
        Pi = qt.Qobj(np.diag([1]*r + [0]*(dx - r)))

    if rho0 is None:
        rho_thermal = qt.thermal_dm(dx, 1)
        D = qt.displace(dx, .5+.5j)
        rho0 = D * rho_thermal * D.dag()

        rho0 = (Pi * rho0 * Pi).unit()
        
    Ix = qt.qeye(dx)
    dims = [[dx, dx], [dx, dx]]
    
    # Term 1: (Ix \otimes Pi) |Omega><Omega| (Ix \otimes Pi)
    Omega = sum(qt.tensor(qt.basis(dx, i), qt.basis(dx, i)) for i in range(dx))
    Omega_proj = Omega * Omega.dag()
    I_tensor_Pi = qt.tensor(Ix, Pi)
    term1 = I_tensor_Pi * Omega_proj * I_tensor_Pi

    # Term 2: (I - Pi^T) \otimes rho0
    term2 = qt.tensor(Ix - Pi.trans(), rho0)

    J = term1 + term2
    J.dims = dims
    
    return J

# Random channel
def J_random(dy, dx):
    
    dim = dy * dx
    
    # 1. Generate a random positive semi-definite matrix H
    G = np.random.randn(dim, dim) + 1j * np.random.randn(dim, dim)
    H = qt.Qobj(G @ G.conj().T, dims=[[dy, dx], [dy, dx]])
    
    # 2. Enforce the TP condition Tr_Y(J) = I_X
    # QuTiP ptrace(1) traces out subsystem 0 (Y) and keeps subsystem 1 (X)
    Hx = H.ptrace(1) 
    
    # Calculate Hx^{-1/2}
    evals, evecs = Hx.eigenstates()
    Hx_inv_half = qt.Qobj(np.zeros((dx, dx)), dims=[[dx], [dx]])
    for val, vec in zip(evals, evecs):
        if val > 1e-12:
            Hx_inv_half += (1.0 / np.sqrt(val)) * vec * vec.dag()
            
    # Apply local transformation I_Y \otimes Hx^{-1/2}
    transform = qt.tensor(qt.qeye(dy), Hx_inv_half)
    J = transform * H * transform
    
    return J


In [ ]:
### ALGORITHM TO MINIMISE EMPIRICAL ERROR

def train_choi(J_initial, gamma, S, M, alpha=.1, epochs=100):
    
    # Extract dimensions from input data
    dx = S[0][0].dims[0][0]
    dy = M[0].dims[0][0]
    dim_yx = dy * dx
    num_states = len(S)
    num_povm = len(M)
    
    # --- Step 2: Pre-computation and Caching ---
    M_rho_cache = np.zeros((num_states, num_povm, dim_yx, dim_yx), dtype=complex)
    p_ij_cache = np.zeros((num_states, num_povm))
    
    for i in range(num_states):
        rho_T = S[i][0].trans()
        for j in range(num_povm):
            M_rho_cache[i, j] = qt.tensor(M[j], rho_T).full()
            p_ij_cache[i, j] = S[i][1][j]
            
    # --- Step 3: CVXPY Projection Setup ---
    J_var = cp.Variable((dim_yx, dim_yx), hermitian=True)
    A_var = cp.Variable((dim_yx, dim_yx), hermitian=True)
    B_var = cp.Variable((dim_yx, dim_yx), hermitian=True)
    
    J_target = cp.Parameter((dim_yx, dim_yx), complex=True)
    
    I_X = np.eye(dx)
    I_YX = np.eye(dim_yx)
    
    constraints = [
        J_var >> 0,
        cp.partial_trace(J_var, dims=[dy, dx], axis=0) == I_X,
        gamma * I_YX - J_var == A_var + cp.partial_transpose(B_var, dims=[dy, dx], axis=0),
        A_var >> 0,
        B_var >> 0
    ]
    
    objective = cp.Minimize(cp.norm(J_var - J_target, 'fro'))
    prob = cp.Problem(objective, constraints)
    
    # --- Step 4: Projected-SGD Loop ---
    J_current = J_initial.full()
    
    for epoch in range(epochs):
        
        # Decay the learning rate
        alpha_t = 0.5 * alpha * (1 + np.cos(np.pi * epoch / epochs))
        
        # Compute predicted probabilities using optimized tensor contraction
        pred_p = np.einsum('ijkl,lk->ij', M_rho_cache, J_current).real
        
        # Evaluate current loss function
        squared_errors = (pred_p - p_ij_cache)**2
        loss = np.mean(np.sum(squared_errors, axis=1))
        
        # Compute exact full-batch gradient 
        grad = np.einsum('ij,ijkl->kl', 2 * (pred_p - p_ij_cache), M_rho_cache) / num_states
        
        # Perform gradient descent step
        J_updated = J_current - alpha_t * grad
        
        # Execute Euclidean projection onto \mathcal C_\gamma
        J_target.value = J_updated
        prob.solve(solver=cp.SCS, eps=1e-5, warm_start=True)
        J_current = J_var.value
        
    return qt.Qobj(J_current, dims=[[dy, dx], [dy, dx]]), loss


In [ ]:
### ALGORITHM TO MAXIMISE EMPIRICAL ERROR DIFFERENCE

def train_choi_diff(gamma, S, M, r, inits=10, alpha=.1, epochs=100):
    
    # Extract dimensions from input data
    dx = S[0][0].dims[0][0]
    dy = M[0].dims[0][0]
    dim_yx = dy * dx
    dim_sub = dy * r
    n = len(S)
    num_povm = len(M)
    
    # --- Step 1: Random Rademacher Assignment ---
    while True:
        mask = np.random.rand(n) >= 0.5
        idx1 = np.where(mask)[0]
        idx2 = np.where(~mask)[0]
        n1, n2 = len(idx1), len(idx2)
        if n1 > 0 and n2 > 0:
            break

    # --- Step 2: Pre-computation and Caching on Rank-r Subspace ---
    # Extract indices for the dy x r subspace (since QuTiP tensor is Y ordered before X)
    idx_r = [y * dx + x for y in range(dy) for x in range(r)]

    M_rho_cache = np.zeros((n, num_povm, dim_sub, dim_sub), dtype=complex)
    p_ij_cache = np.zeros((n, num_povm))
    
    for i in range(n):
        rho_T = S[i][0].trans()
        for j in range(num_povm):
            full_mat = qt.tensor(M[j], rho_T).full()
            # Slice the strictly active subspace
            M_rho_cache[i, j] = full_mat[np.ix_(idx_r, idx_r)]
            p_ij_cache[i, j] = S[i][1][j]
            
    M_rho_1, p_ij_1 = M_rho_cache[idx1], p_ij_cache[idx1]
    M_rho_2, p_ij_2 = M_rho_cache[idx2], p_ij_cache[idx2]
            
    # --- Step 3: CVXPY Projection Setup (Subspace Restricted) ---
    J_var = cp.Variable((dim_sub, dim_sub), hermitian=True)
    A_var = cp.Variable((dim_sub, dim_sub), hermitian=True)
    B_var = cp.Variable((dim_sub, dim_sub), hermitian=True)
    
    J_target = cp.Parameter((dim_sub, dim_sub), complex=True)
    
    I_X_sub = np.eye(r)
    I_YX_sub = np.eye(dim_sub)
    
    constraints = [
        J_var >> 0,
        cp.partial_trace(J_var, dims=[dy, r], axis=0) == I_X_sub,
        gamma * I_YX_sub - J_var == A_var + cp.partial_transpose(B_var, dims=[dy, r], axis=0),
        A_var >> 0,
        B_var >> 0
    ]
    
    objective = cp.Minimize(cp.norm(J_var - J_target, 'fro'))
    prob = cp.Problem(objective, constraints)
    
    # --- Step 4: Multi-Initialization Projected-SGD Loop ---
    best_target_obj = -np.inf
    best_J_full = None
        
    for init in range(inits):
        # 1. Pick a random initial Choi matrix
        J_start_full = J_random(dy, dx).full()
            
        # Extract purely the subspace block for optimization
        J_current = J_start_full[np.ix_(idx_r, idx_r)]
        
        for epoch in range(epochs):
            alpha_t = 0.5 * alpha * (1 + np.cos(np.pi * epoch / epochs))
            
            pred_p1 = np.einsum('ijkl,lk->ij', M_rho_1, J_current).real
            pred_p2 = np.einsum('ijkl,lk->ij', M_rho_2, J_current).real
            
            loss_1 = np.mean(np.sum((pred_p1 - p_ij_1)**2, axis=1))
            loss_2 = np.mean(np.sum((pred_p2 - p_ij_2)**2, axis=1))
            target_obj = loss_1 - loss_2
            
            grad_1 = np.einsum('ij,ijkl->kl', 2 * (pred_p1 - p_ij_1), M_rho_1) / n1
            grad_2 = np.einsum('ij,ijkl->kl', 2 * (pred_p2 - p_ij_2), M_rho_2) / n2
            grad_opt = grad_2 - grad_1
            
            J_updated = J_current - alpha_t * grad_opt
            
            J_target.value = J_updated
            prob.solve(solver=cp.SCS, eps=1e-5, warm_start=True)
            J_current = J_var.value
            
        # 2. Retain the maximum generalized discrepancy encountered
        if target_obj > best_target_obj:
            best_target_obj = target_obj
            # Re-embed the optimized submatrix back into the full dimensional space
            best_J_full = np.zeros((dim_yx, dim_yx), dtype=complex)
            best_J_full[np.ix_(idx_r, idx_r)] = J_current

        print(f"{'Completed initialisation'} {init+1} {'out of'} {inits}{'.'}")
            
    return qt.Qobj(best_J_full, dims=[[dy, dx], [dy, dx]]), best_target_obj


In [ ]:
### GENERATE DATA FOR EXPERIMENTS

# Parameters for generating data; tweak for different settings
dx = 4 # input dimension (before CPTP)
dy = 4 # output dimension (after CPTP)
r = 4 # compression parameter
num_states = 500 # sample size
num_povm = 8 # number of elements in POVM
num_meas = 0 # number of measurements to approximate the Born probabilities; for true probabilities set to: 0

# Pick underlying channel
J_true = J_random(dy,dx)

# Generate random POVM
random_pos_ops = [qt.rand_dm(dy) for _ in range(num_povm)]

# Sum them to find the normalization operator
total_sum = sum(random_pos_ops)

# Diagonalize and compute the inverse square root
evals, evecs = total_sum.eigenstates()
inv_sqrt_op = sum([(1 / np.sqrt(val)) * vec * vec.dag() for val, vec in zip(evals, evecs)])

# Construct the POVM elements M_j
M = [inv_sqrt_op * P * inv_sqrt_op for P in random_pos_ops]

### Generate training data

# Generate an empty set
S = []

# Generate initial dataset with rhos
rhos = [evol(J_compression(dx, r), qt.rand_dm(dx)) for _ in range(num_states)]

# Generate probability vectors and form dataset
for rho in rhos:
    rho_out = evol(J_true, rho)

    probs = np.array([(M_j * rho_out).tr().real for M_j in M])
    probs = np.clip(probs, 0, None)
    probs /= np.sum(probs)

    if num_meas == 0:
        p = probs
    else:
        counts = np.random.multinomial(int(num_meas), probs)
        p = counts / float(num_meas)

    S.append((rho, p))

# Remove intermediate variable
del rhos


In [ ]:
### RUN TRAINING ALGORITHM TO MINIMISE EMPIRICAL ERROR

# Parameters for experiments; tweak for different settings
gamma = .3 # regularisation parameter
alpha = 100 # initial learning rate
epochs = 50 # number of epochs
split = .8 # train/test split parameter
N = 10 # number of repetitions

train_losses = []
test_losses = []

for _ in range(N):
    # Shuffle S before splitting
    S_shuffled = S.copy()
    np.random.shuffle(S_shuffled)

    # Split data into train and test sets
    num_train = int(len(S_shuffled) * split)
    S_train = S_shuffled[:num_train]
    S_test = S_shuffled[num_train:]

    # Initialize random Choi matrix
    J_initial = J_random(dy, dx)

    # Train on S_train
    J_trained, train_loss = train_choi(J_initial, gamma, S_train, M, alpha=alpha, epochs=epochs)

    # Evaluate on S_test
    M_rho_cache_test = np.array([
        [qt.tensor(M[j], S_test[i][0].trans()).full() for j in range(len(M))]
        for i in range(len(S_test))
    ])
    p_ij_cache_test = np.array([S_test[i][1] for i in range(len(S_test))])

    pred_p_test = np.einsum('ijkl,lk->ij', M_rho_cache_test, J_trained.full()).real
    test_loss = np.mean(np.sum((pred_p_test - p_ij_cache_test)**2, axis=1))

    train_losses.append(train_loss)
    test_losses.append(test_loss)

# Calculate averages across N runs
avg_train_loss = np.mean(train_losses)
avg_test_loss = np.mean(test_losses)

# Create output list
outputs = f"{dx} {dy} {r} {num_states} {split} {num_povm} {num_meas} {gamma} {alpha} {epochs} {N} {avg_train_loss:.5f} {avg_test_loss:.5f}"

# Print output list
print(outputs)


In [ ]:
### RUN TRAINING ALGORITHM TO MAXIMISE EMPIRICAL ERROR DIFFERENCE (figure 1 of paper)

# Parameters for experiments; tweak for different settings
gamma = .5 # regularisation parameter
alpha = 100 # iniital learning rate
epochs = 50 # number of epochs
trials = 10 # number of random partitions of the training set
inits = 10 # number of random initialisations of the SGD

# Initialise empty list of errors
errors = []

# Run trials for different random partitions
for trial in range(trials):
    target_obj = train_choi_diff(gamma, S, M, r, inits, alpha, epochs)[1]
    errors.append(target_obj)
    print(f"{'Completed trial'} {trial+1} {'out of'} {trials}{'.'}")

# Calculate mean and standard deviation of the generalisation error gap
m_error = np.mean(errors)
sd_error = np.std(errors)

# Create output list
outputs = f"{dx} {dy} {r} {num_states} {num_povm} {num_meas} {gamma} {alpha} {epochs} {trials} {inits} {m_error:.5f} {sd_error:.5f}"

# Print output list
print(outputs)
